# Notebook 5 — CNN on Rice Images (R4)
Rice Variety Classification | F20DL 2026/2027

**Dataset:** Kaggle Rice Image Dataset — 5 classes, 75,000 images

To download:
```bash
# 1. Place your kaggle.json in ~/.kaggle/kaggle.json
# 2. Run:
mkdir -p ../data/images
.venv/bin/kaggle datasets download -d muratkokludataset/rice-image-dataset -p ../data/images --unzip
```

The images should end up at `../data/images/Rice_Image_Dataset/` with one folder per class.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score
import pandas as pd

torch.manual_seed(42)

DATA_DIR  = '../data/images/Rice_Image_Dataset'
IMG_SIZE  = 64
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
n = len(full_dataset)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
train_idx = indices[:int(0.70 * n)]
val_idx   = indices[int(0.70 * n):int(0.85 * n)]
test_idx  = indices[int(0.85 * n):]

test_dataset = datasets.ImageFolder(DATA_DIR, transform=test_transform)

train_dl = DataLoader(Subset(full_dataset, train_idx),  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(Subset(full_dataset, val_idx),    batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_dl  = DataLoader(Subset(test_dataset, test_idx),   batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Classes: {CLASS_NAMES}')
print(f'Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')

## 0. Sample Images

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(15, 3))
for ax, cls in zip(axes, CLASS_NAMES):
    sample_ds = datasets.ImageFolder(DATA_DIR, transform=test_transform)
    idx = next(i for i, (_, label) in enumerate(sample_ds.samples) if sample_ds.classes[label] == cls)
    img, _ = sample_ds[idx]
    img_np = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(img_np.clip(0, 1))
    ax.set_title(cls)
    ax.axis('off')
plt.suptitle('Sample Images per Class', fontsize=13)
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=100)
plt.show()

## 1. Define CNN

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

EPOCHS = 20
train_losses, val_losses, val_accs = [], [], []
best_val_acc, best_state = 0, None

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_dl))

    model.eval()
    v_loss, correct = 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            v_loss += criterion(out, yb).item()
            correct += (out.argmax(1) == yb).sum().item()
    val_losses.append(v_loss / len(val_dl))
    val_accs.append(correct / len(val_idx))
    scheduler.step(val_losses[-1])

    if val_accs[-1] > best_val_acc:
        best_val_acc = val_accs[-1]
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch+1:2d}: train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}  val_acc={val_accs[-1]:.4f}')

print(f'\nBest val accuracy: {best_val_acc:.4f}')
model.load_state_dict(best_state)

## 2. Train

In [ ]:
# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set(title='CNN Loss Curves', xlabel='Epoch', ylabel='Loss')
ax1.legend()
ax2.plot(val_accs, color='green')
ax2.set(title='Validation Accuracy', xlabel='Epoch', ylabel='Accuracy')
plt.tight_layout()
plt.savefig('../results/cnn_training.png', dpi=100)
plt.show()

# Test evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        all_preds.extend(model(xb).argmax(1).cpu().numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(all_labels, all_preds, display_labels=CLASS_NAMES, ax=ax, colorbar=False)
ax.set_title('CNN Confusion Matrix')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../results/cnn_confusion.png', dpi=100)
plt.show()

cnn_acc = accuracy_score(all_labels, all_preds)
cnn_f1  = f1_score(all_labels, all_preds, average='weighted')
print(f'\nCNN Test Accuracy: {cnn_acc:.4f}  F1: {cnn_f1:.4f}')
torch.save(model.state_dict(), '../models/cnn.pt')

## 3. Plot & Save

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Training Loss')
ax2.plot(val_accs); ax2.set_title('Validation Accuracy')
plt.tight_layout(); plt.savefig('../results/cnn_training.png', dpi=100); plt.show()
torch.save(model.state_dict(), '../models/cnn.pt')